# RQ-VAE Training & Evaluation

Train a Residual Quantized VAE to assign hierarchical semantic IDs to product embeddings.

**Pipeline:** Config → Train → Evaluate → Export

In [ ]:
import sys
from pathlib import Path

# pipeline/ is the working directory — rqvae/ is a sibling package
sys.path.insert(0, str(Path(".").resolve()))

import torch
from rqvae import (
    RQVAEConfig, RQVAE, TrainState, train_rqvae, prepare_data, set_seed,
    setup_logger, get_device_info,
)

setup_logger("train-rqvae", log_to_file=True)

# --- Config ---
EMBEDDINGS = Path("..").resolve() / "data/embeds/Pet_Supplies_items_with_embeddings.parquet"
CHECKPOINT_DIR = Path("..").resolve() / "models/rqvae"

config = RQVAEConfig(
    embeddings_path=EMBEDDINGS,
    checkpoint_dir=CHECKPOINT_DIR,
    batch_size=4096,
    num_epochs=1000,
    steps_per_train_log=10,
    steps_per_val_log=200,
    use_kmeans_init=True,
    val_split=0.05,
    seed=42,
)
config.validate()
set_seed(config.seed)

# --- Device & Data ---
device_info = get_device_info()
device = device_info.device

if device in {"cpu", "mps"}:
    config.batch_size = 1024
    config.gradient_accumulation_steps = 4

train_loader, val_loader = prepare_data(config.embeddings_path, config, device_info)
model = RQVAE(config).to(device)

print(f"Device: {device} | Batch: {config.batch_size} | Epochs: {config.num_epochs}")

In [ ]:
# --- Train ---
state = train_rqvae(model=model, data_loader=train_loader, val_loader=val_loader,
                    config=config, device=device, state=TrainState())

# Save final model
final_path = CHECKPOINT_DIR / "final_model.pth"
torch.save({
    "model_state_dict": model.state_dict(),
    "config": {k: (str(v) if isinstance(v, Path) else v) for k, v in config.__dict__.items()},
    "epoch": state.epoch, "step": state.global_step, "best_loss": state.best_loss,
}, final_path)
print(f"Final model: {final_path} (loss={state.best_loss:.4e})")

In [ ]:
# --- Evaluate ---
from rqvae import evaluate_rqvae, print_report, load_embeddings

embeddings = load_embeddings(EMBEDDINGS)
report = evaluate_rqvae(model, embeddings, codebook_size=config.codebook_size)
print_report(report)

In [ ]:
# --- Export semantic IDs ---
import numpy as np
import polars as pl
from rqvae.evaluate import encode_dataset

semantic_ids, _ = encode_dataset(model, embeddings)

df = pl.read_parquet(EMBEDDINGS)
df = df.with_columns(pl.Series("semantic_ids", semantic_ids.tolist()))

out_path = str(EMBEDDINGS).replace(".parquet", "_with_semantic_ids.parquet")
df.write_parquet(out_path)
print(f"Exported: {out_path} ({len(df):,} items, {semantic_ids.shape[1]} levels)")